In [ ]:
import itertools
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier

# -----------------------------
# 1) CONFIG: choose features to search over
# -----------------------------
candidate_features = [
    "sex", "age", "pclass", "embarked", "fare_log",
    "family_size", "alone", "title", "deck", "has_cabin"
]

# Categorical columns (one-hot encode if included)
categorical_features = ["sex", "pclass", "embarked", "title", "deck"]

# Target
y = df_titanic["survived"].astype(int)

# Choose model for simulation:
# - DecisionTreeClassifier: good for interpretability (usually lower ceiling)
# - GradientBoostingClassifier: often higher accuracy
MODEL_NAME = "tree"  # "tree" or "gb"

def make_model(model_name="gb", random_state=42):
    if model_name == "tree":
        return DecisionTreeClassifier(
            max_depth=7,
            min_samples_split=7,
            min_samples_leaf=7,
            random_state=random_state
        )
    else:
        return GradientBoostingClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            random_state=random_state
        )

# -----------------------------
# 2) Evaluation function (always returns accuracy)
# -----------------------------
def eval_accuracy(X, y, model_name="gb", test_size=0.2, random_state=42, stratify=True):
    model = make_model(model_name=model_name, random_state=random_state)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y if stratify else None
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    return accuracy_score(y_test, pred)

# -----------------------------
# 3) Run all possible combinations
# -----------------------------
results = []

for r in range(1, len(candidate_features) + 1):
    for combo in itertools.combinations(candidate_features, r):
        cols = list(combo)

        # Build X
        X = df_titanic[cols].copy()

        # One-hot encode only categoricals that are present
        encoding_cols = [c for c in categorical_features if c in cols]
        if encoding_cols:
            X = pd.get_dummies(X, columns=encoding_cols, drop_first=True)

        # Handle any missing values quickly (optional)
        # If you already cleaned/imputed earlier, you can remove this block.
        if X.isna().any().any():
            # numeric fill with median
            for c in X.columns:
                if X[c].dtype != "object":
                    X[c] = X[c].fillna(X[c].median())
            # if any leftover NaNs, fill with 0
            X = X.fillna(0)

        # Evaluate
        acc = eval_accuracy(X, y, model_name=MODEL_NAME, test_size=0.2, random_state=42, stratify=True)

        results.append({
            "cols": cols,
            "encoding_cols": encoding_cols,
            "n_raw_features": len(cols),
            "n_model_features": X.shape[1],
            "accuracy": acc
        })

# Results dataframe
results_df = pd.DataFrame(results).sort_values("accuracy", ascending=False).reset_index(drop=True)

# Show top 20
print(results_df.head(20))

# Print best combo in your exact format
best = results_df.iloc[0]
print("\nBEST COMBINATION\n")
print(f"cols = {list(best['cols'])}")
print(f"encoding_cols = {best['encoding_cols']}")
print("X = df_titanic[cols]")
print("X = pd.get_dummies(X, columns=encoding_cols, drop_first=True)")
print("accuracy_test(X, y)  # or eval_accuracy(X, y)")
print("\nBest Accuracy:", best["accuracy"])